## Bloc 1 : Imports

In [ ]:
import os, pickle, re
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.cluster import MiniBatchKMeans
from sentence_transformers import SentenceTransformer
import torch
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

## Bloc 2 : Chargement CSV + split client/company

In [ ]:
CSV_PATH = '../Projet/twcs.csv'
DATA_DIR = '../data/'
os.makedirs(DATA_DIR, exist_ok=True)

df_raw = pd.read_csv(CSV_PATH, dtype=str)
df_raw['inbound'] = df_raw['inbound'].map({'True': True, 'False': False})

df_client  = df_raw[df_raw['inbound'] == True].copy()

print(f'Total brut : {len(df_raw):,}')
print(f'Client     : {len(df_client):,}')


## Bloc 3 : Diagnostic AVANT nettoyage

In [ ]:
for df, name in [(df_client, 'CLIENT')]:
    nan_text     = df['text'].isna().sum()
    nan_id       = df['tweet_id'].isna().sum()
    dup_id       = df.duplicated(subset='tweet_id').sum()
    empty_text   = (df['text'].str.strip().str.len() < 3).sum()

    print(f'\n=== {name} ({len(df):,} tweets) ===')
    print(f'  NaN text       : {nan_text}')
    print(f'  NaN tweet_id   : {nan_id}')
    print(f'  Doublons id    : {dup_id}')
    print(f'  Texte vide/court : {empty_text}')
    print(f'  → à supprimer  : ~{nan_text + nan_id + dup_id + empty_text}')


## Bloc 4 : structural_clean + application

In [ ]:
def structural_clean(df, name):
    n0 = len(df)
    #df = df.dropna(subset=['text', 'tweet_id'])
    #df = df.drop_duplicates(subset='tweet_id')
    df = df[df['text'].str.strip().str.len() >= 3]
    print(f'{name} : {n0:,} → {len(df):,} ({n0 - len(df):,} supprimés)')
    return df.reset_index(drop=True)

df_client  = structural_clean(df_client,  'df_client')


## Bloc 5 : clean_tweet + application

Deux niveaux de nettoyage clairs :

clean_tweet : pour l'embedding — enlève @mentions, URLs, chiffres, ponctuation, met en minuscules. Garde les stopwords pour que le modèle comprenne la phrase.
Pour les mots-clés affichés par cluster : on filtre les stopwords au moment de l'affichage uniquement.

In [ ]:
STOP = set(stopwords.words('english'))

def clean_tweet_intent(text):
    text = re.sub(r'@\w+', '', text)               # @mentions → bruit de marque
    text = re.sub(r'http\S+', '', text)             # URLs
    text = re.sub(r'#\w+', '', text)                # hashtags
    text = re.sub(r'\d+', '', text)                 # chiffres
    text = re.sub(r"[^\w\s?!']", ' ', text)         # ponctuation sauf ? ! '
    text = re.sub(r'\s+', ' ', text)                # espaces multiples
    return text.strip().lower()


df_client['text_clean']  = df_client['text'].apply(clean_tweet_intent)

for i in range(10):
    print(f'[{i}] Avant : {df_client["text"].iloc[i]}')
    print(f'    Après : {df_client["text_clean"].iloc[i]}')
    print()

## Bloc 6 : Config + embeddings

In [ ]:
SAMPLE_CLUSTER = 200_000
K_MIN          = 4
K_MAX          = 14
RANDOM_STATE   = 42
BATCH_SIZE     = 256

assert torch.cuda.is_available(), "GPU non détecté — vérifier CUDA"
device = torch.device('cuda')
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


embed_model = SentenceTransformer(
    'cardiffnlp/twitter-roberta-base',
    model_kwargs={'use_safetensors': True},
    device='cuda'
)


# Échantillonnage
sample_cl = df_client.sample(min(SAMPLE_CLUSTER, len(df_client)), random_state=RANDOM_STATE)

print('Encodage client...')
emb_cl = embed_model.encode(sample_cl['text_clean'].tolist(),
                             batch_size=BATCH_SIZE, show_progress_bar=True,
                             convert_to_numpy=True).astype(np.float32)


## Bloc 7 : KMeans + mots-clés par cluster (client)

In [ ]:
STOP_INTENT = STOP - {'not', 'no', 'never', 'please', 'still', 'when', 'how',
                      'what', 'why', 'where', 'help', 'cant', 'wont', 'dont'}

def show_clusters(sample_df, emb, K, name):
    km = MiniBatchKMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=15)
    sample_df = sample_df.copy()
    sample_df['cluster'] = km.fit_predict(emb)

    print(f'\n{"="*60}')
    print(f'{name} — K={K}')
    print(f'{"="*60}')
    for c in range(K):
        mask = sample_df['cluster'] == c
        subset = sample_df[mask]

        # Mots-clés sans les stopwords génériques (mais garde les marqueurs d'intention)
        words = [w for t in subset['text_clean']
                   for w in t.split()
                   if w not in STOP_INTENT]
        top5 = [w for w, _ in Counter(words).most_common(5)]

        # Signaux d'intention
        pct_q  = subset['text_clean'].str.contains(r'\?').mean() * 100
        pct_ex = subset['text_clean'].str.contains(r'!').mean()  * 100
        avg_len = subset['text_clean'].str.split().str.len().mean()

        examples = subset['text'].sample(min(3, mask.sum()), random_state=42).tolist()

        print(f'\nC{c:2d} ({mask.sum():>6,} tweets) → {top5}')
        print(f'     ? {pct_q:.0f}%  ! {pct_ex:.0f}%  len moy {avg_len:.0f}')
        for ex in examples:
            print(f'     > {ex[:100]}')
    return km, sample_df

for k in range(K_MIN, K_MAX + 1):
    show_clusters(sample_cl, emb_cl, k, 'CLIENT')

## Bloc 8 : Labellisation par mots-clés

In [ ]:
def assign_label(text):
    words = set(text.lower().split())
    text_lower = text.lower()

    # Non-anglais : scripts non-latins OU ≥2 mots romans
    if sum(1 for c in text if ord(c) > 0x3000) > 3:
        return 'non_english'
    if len(words & {'que','pas','por','con','pour','vous','para','de','la','en','les','du',
                'ich','die','der','das','und','ist',
                'per','con','non','una','sono','che',
                'est','une','sur','avec','mais','tout',
                'yang','untuk','dengan','tidak','saya'}) >= 2:
        return 'non_english'

    # Accusé réception : court + merci (les deux clusters ack fusionnés)
    if len(words) <= 10 and words & {'thanks','thank','sent','done','yes','ok','noted','received'}:
        return 'acknowledgment'

    # Question : ? présent + mot interrogatif (cluster ? 40-53%)
    if '?' in text and words & {'when','how','what','where','why','which'}:
        return 'question'

    # Problème technique (cluster phone/app, ? 30-35%)
    problem_seq = ['not working',"doesn't work",'broken','error','crash',
                   'frozen','bug','glitch','failed','unable','not loading']
    if any(s in text_lower for s in problem_seq):
        return 'problem_report'

    # Plainte (cluster flight/!, mots négatifs forts)
    if words & {'worst','terrible','awful','unacceptable','horrible',
                'ridiculous','shame','disappointed','useless','rude','disgrace'}:
        return 'complaint'

    # Relance
    follow_seq = ['still waiting','any update','still no','no response','been waiting']
    if any(s in text_lower for s in follow_seq):
        return 'follow_up'

    # Demande d'aide explicite
    help_seq = ['please help','help me','need help','can you help','please fix','please check']
    if any(s in text_lower for s in help_seq):
        return 'help_request'

    return 'general'

df_client = df_client[df_client['text_clean'].str.strip().str.len() >= 3].reset_index(drop=True)
df_client['label'] = df_client['text_clean'].apply(assign_label)
print(df_client['label'].value_counts())



## Bloc 9 : Analyse / validation

In [ ]:
labels = df_client['label'].unique()

for lbl in sorted(labels):
    subset = df_client[df_client['label'] == lbl]
    pct    = len(subset) / len(df_client) * 100
    print(f'\n{"="*60}')
    print(f'{lbl} — {len(subset):,} tweets ({pct:.1f}%)')
    print(f'{"="*60}')
    for _, row in subset.sample(min(5, len(subset)), random_state=42).iterrows():
        print(f'  > {row["text"][:100]}')

print(f'\nTotal : {len(df_client):,} tweets')
print(f'general : {(df_client["label"] == "general").sum() / len(df_client) * 100:.1f}% non couverts')

## Bloc 10 : Équilibrage

In [ ]:
N_MAX = 100_000

df_bal = (df_client.groupby('label', group_keys=False)
          .apply(lambda x: x.sample(min(len(x), N_MAX), random_state=RANDOM_STATE)))
df_bal = df_bal.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(df_bal['label'].value_counts())

## Bloc 11 : Split stratifié 70/15/15

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df_bal['text_clean'].values, df_bal['label'].values

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE)

print(f'Train : {len(X_tr):,} | Val : {len(X_val):,} | Test : {len(X_te):,}')

## Bloc 12 : Vocabulaire

In [ ]:
counter = Counter()
for text in X_tr:
    counter.update(text.split())

VOCAB_SIZE = 15_000
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print(f'Vocab : {len(vocab):,} tokens')

## Bloc 13 : Tokenisation

In [ ]:
MAX_LEN = 64

LABEL_NAMES = sorted(df_bal['label'].unique())
label2id    = {l: i for i, l in enumerate(LABEL_NAMES)}
id2label    = {i: l for l, i in label2id.items()}

def tokenize(text):
    tokens = text.split()[:MAX_LEN]
    ids = [vocab.get(w, vocab['<UNK>']) for w in tokens]
    return ids if ids else [vocab['<UNK>']]

client_train = [(tokenize(t), label2id[l]) for t, l in zip(X_tr,  y_tr)]
client_val   = [(tokenize(t), label2id[l]) for t, l in zip(X_val, y_val)]
client_test  = [(tokenize(t), label2id[l]) for t, l in zip(X_te,  y_te)]


## Bloc 14 : Sauvegarde

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

with open(DATA_DIR + 'vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
with open(DATA_DIR + 'label_maps.json', 'w') as f:
    json.dump({'client': id2label}, f, indent=2)
with open(DATA_DIR + 'client_data.pkl', 'wb') as f:
    pickle.dump({'train': client_train, 'val': client_val, 'test': client_test}, f)

# Textes bruts pour DistilBERT
text_train = [(t, label2id[l]) for t, l in zip(X_tr, y_tr)]
text_val   = [(t, label2id[l]) for t, l in zip(X_val, y_val)]
text_test  = [(t, label2id[l]) for t, l in zip(X_te, y_te)]
with open(DATA_DIR + 'client_text_data.pkl', 'wb') as f:
    pickle.dump({'train': text_train, 'val': text_val, 'test': text_test}, f)

print('Sauvegardé :', DATA_DIR)
print(f'  N_CLASSES = {len(LABEL_NAMES)} — {LABEL_NAMES}')


## Bloc 15 : Vérification

In [ ]:
import os
for f in ['vocab.pkl','label_maps.json','client_data.pkl','client_text_data.pkl']:
    size = os.path.getsize(f'../data/{f}') / 1e6
    print(f'{f:30s} {size:.1f} MB')
